# HotpotQA LLM Semantic Prediction Compare

这个 notebook 用 `director` 作为目标词，直接复用 HotpotQA scan cache 里的样本文本，并比较多个 LLM 在同一批样本上的语义预测结果。

要点：
- 样本文本提取方式与 `hotpotqa_fft_span_compare.ipynb` 保持一致。
- 这里显式关闭 LLM 语义预测缓存，不会直接复用旧的 SQLite 查询结果。
- 如果启用 LLM-driven Wikidata candidate filtering，也默认关闭它的缓存，避免候选集合直接命中旧结果。
- 结果会输出每个模型的预测表、pairwise agreement，以及所有模型存在分歧的样本。

In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")

Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from itertools import combinations
from pathlib import Path
import pickle

import pandas as pd
from IPython.display import display

from text_processing import normalize_text
from utils import (
    build_wikidata_candidate_bank as shared_build_wikidata_candidate_bank,
    load_wikidata_definition_candidates as shared_load_wikidata_definition_candidates,
)
from llm_semantic_labeler import (
    DEFAULT_CONTEXT_WORD_WINDOW as DEFAULT_LLM_CONTEXT_WORD_WINDOW,
    DEFAULT_MAX_TOKENS as DEFAULT_LLM_MAX_TOKENS,
    SYSTEM_PROMPT,
    USER_PROMPT_TEMPLATE,
    _candidate_block,
    _extract_json_object,
    _normalize_space,
    _parse_selected_index,
    build_context_signature,
)
from local_llm import LocalLLMClient, LocalLLMConfig

HOTPOT_SCAN_STORE_PATH = Path("hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl")
DEFAULT_CONTEXT_MODE = "sentence_neighbors"
DEFAULT_QUERY_KIND = None

pd.set_option("display.max_colwidth", None)


def load_hotpot_scan_store(scan_store_path: Path = HOTPOT_SCAN_STORE_PATH):
    if not scan_store_path.exists():
        raise FileNotFoundError(
            f"HotpotQA scan store not found at {scan_store_path}. Please prepare the scan cache first."
        )
    with scan_store_path.open("rb") as handle:
        store = pickle.load(handle)
    print(f"Loaded scan-only store from {scan_store_path}")
    print(store["stats"])
    return store


def lookup_records(store, query_text, kind=None, include_text=True, include_cleaned_text=True):
    normalized_query = normalize_text(query_text.strip())
    records = list(store["index"].get(normalized_query, []))

    if kind is not None:
        records = [record for record in records if record["kind"] == kind]

    if not include_text and not include_cleaned_text:
        return records

    enriched_records = []
    for record in records:
        item = dict(record)
        document_idx = item["document_idx"]
        if include_text:
            item["text"] = store["documents"][document_idx]["text"]
        if include_cleaned_text:
            item["cleaned_text"] = store["cleaned_documents"][document_idx]
        enriched_records.append(item)
    return enriched_records


def _find_left_boundary(text: str, index: int) -> int:
    return max(text.rfind(".", 0, index), text.rfind("!", 0, index), text.rfind("?", 0, index))


def _find_right_boundary(text: str, index: int) -> int:
    right_candidates = [text.find(".", index), text.find("!", index), text.find("?", index)]
    right_candidates = [idx for idx in right_candidates if idx != -1]
    return len(text) if not right_candidates else min(right_candidates) + 1


def _build_context_from_bounds(cleaned_text, span, context_start, context_end):
    start_char, end_char = span
    context_raw = cleaned_text[context_start:context_end]

    if not context_raw.strip():
        context_start = max(0, start_char - 120)
        context_end = min(len(cleaned_text), end_char + 120)
        context_raw = cleaned_text[context_start:context_end]

    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - context_start - left_trim
    local_end = end_char - context_start - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def extract_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = _find_left_boundary(cleaned_text, start_char)
    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = _find_right_boundary(cleaned_text, end_char)
    return _build_context_from_bounds(cleaned_text, span, context_start, context_end)


def extract_neighbor_sentence_context(cleaned_text, span):
    start_char, end_char = span
    left_boundary = _find_left_boundary(cleaned_text, start_char)
    context_start = 0 if left_boundary == -1 else left_boundary + 1
    context_end = _find_right_boundary(cleaned_text, end_char)

    if context_start > 0:
        previous_boundary = _find_left_boundary(cleaned_text, max(0, context_start - 1))
        context_start = 0 if previous_boundary == -1 else previous_boundary + 1

    if context_end < len(cleaned_text):
        context_end = _find_right_boundary(cleaned_text, context_end)

    return _build_context_from_bounds(cleaned_text, span, context_start, context_end)


def extract_full_context(cleaned_text, span):
    start_char, end_char = span
    context_raw = cleaned_text
    left_trim = len(context_raw) - len(context_raw.lstrip())
    context_text = context_raw.strip()

    local_start = start_char - left_trim
    local_end = end_char - left_trim
    local_start = max(0, min(local_start, len(context_text)))
    local_end = max(local_start, min(local_end, len(context_text)))

    return {
        "context_text": context_text,
        "local_span": (local_start, local_end),
    }


def extract_prompt_context(cleaned_text, span, prompt_context_mode="sentence"):
    if prompt_context_mode == "sentence":
        return extract_sentence_context(cleaned_text, span)
    if prompt_context_mode == "full_text":
        return extract_full_context(cleaned_text, span)
    if prompt_context_mode == "sentence_neighbors":
        return extract_neighbor_sentence_context(cleaned_text, span)
    raise ValueError(
        f"Unsupported prompt_context_mode={prompt_context_mode!r}. Use 'sentence', 'sentence_neighbors', or 'full_text'."
    )


def build_hotpot_prompt(
    record,
    query_text,
    mark_target=False,
    left_marker="[TGT]",
    right_marker="[/TGT]",
    prompt_context_mode="sentence",
):
    context_info = extract_prompt_context(
        record["cleaned_text"],
        record["span"],
        prompt_context_mode=prompt_context_mode,
    )
    context_text = context_info["context_text"]
    local_start, local_end = context_info["local_span"]
    prompt_context = context_text

    if mark_target:
        prompt_context = (
            f"{context_text[:local_start]}{left_marker} {context_text[local_start:local_end]} {right_marker}{context_text[local_end:]}"
        )

    prompt_text = (
        f"Context: {prompt_context}\n"
        f"Target word: {query_text.strip()}\n\n"
        f'Question: What does "{query_text.strip()}" mean in this context?'
    )

    return {
        "context_text": context_text,
        "matched_text": context_text[local_start:local_end],
        "local_span": (local_start, local_end),
        "prompt_text": prompt_text,
    }


def collect_query_span_records(
    store,
    query_text,
    kind=None,
    prompt_context_mode=DEFAULT_CONTEXT_MODE,
    mark_target=False,
    max_records=None,
):
    query_records = lookup_records(
        store,
        query_text,
        kind=kind,
        include_text=True,
        include_cleaned_text=True,
    )

    if not query_records:
        raise ValueError(f"No records found for span={query_text!r}.")

    if max_records is not None:
        query_records = query_records[: int(max_records)]

    records = []
    for record in query_records:
        prompt_info = build_hotpot_prompt(
            record,
            query_text,
            mark_target=mark_target,
            prompt_context_mode=prompt_context_mode,
        )
        item = dict(record)
        item["matched_text"] = prompt_info["matched_text"]
        item["context_text"] = prompt_info["context_text"]
        item["local_span"] = prompt_info["local_span"]
        item["prompt_text"] = prompt_info["prompt_text"]
        item["record_index"] = len(records)
        records.append(item)

    return {
        "query_text": query_text,
        "kind": kind,
        "record_count": len(records),
        "prompt_context_mode": prompt_context_mode,
        "mark_target": mark_target,
        "max_records": max_records,
        "records": records,
    }


def load_wikidata_definition_candidates(
    query_text: str,
    use_detailed_description: bool = True,
    exact_match_text: bool = False,
    filter_name: bool = True,
    require_detailed_description: bool = False,
    candidate_limit: int = 5,
    use_llm_filter: bool = False,
    llm_filter_use_api: bool = False,
    llm_filter_use_cache: bool = True,
):
    return shared_load_wikidata_definition_candidates(
        query_text,
        use_detailed_description=use_detailed_description,
        exact_match_text=exact_match_text,
        limit=int(candidate_limit),
        filter_name=filter_name,
        require_detailed_description=require_detailed_description,
        target_candidate_count=int(candidate_limit) if use_llm_filter else None,
        use_llm_filter=use_llm_filter,
        llm_filter_use_api=llm_filter_use_api,
        llm_filter_use_cache=llm_filter_use_cache,
    )


def build_wikidata_candidate_bank(candidates_df: pd.DataFrame, definition_column: str):
    return shared_build_wikidata_candidate_bank(candidates_df, definition_column)

In [3]:
def choose_wikidata_candidate_with_llm_no_cache(
    *,
    span_text: str,
    context_text: str,
    candidate_bank,
    matched_text: str | None = None,
    local_span: tuple[int, int] | None = None,
    context_word_window: int = DEFAULT_LLM_CONTEXT_WORD_WINDOW,
    config: LocalLLMConfig | None = None,
    client: LocalLLMClient | None = None,
    temperature: float = 0.0,
    max_tokens: int = DEFAULT_LLM_MAX_TOKENS,
):
    candidate_bank_list = [dict(candidate) for candidate in candidate_bank]
    if not candidate_bank_list:
        raise ValueError("candidate_bank must be non-empty.")

    llm_client = client or LocalLLMClient(config)
    active_config = llm_client.config
    context_signature = build_context_signature(
        span_text,
        context_text,
        local_span=local_span,
        context_word_window=context_word_window,
    )

    prompt = USER_PROMPT_TEMPLATE.format(
        span_text=_normalize_space(span_text),
        context_text=_normalize_space(context_text),
        matched_text=_normalize_space(matched_text or span_text),
        candidate_block=_candidate_block(candidate_bank_list),
    )
    raw_response = llm_client.complete(
        prompt,
        system_prompt=SYSTEM_PROMPT,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    payload = _extract_json_object(raw_response)
    selected_index = _parse_selected_index(payload, candidate_bank_list)
    selected_candidate = dict(candidate_bank_list[selected_index])
    reason = payload.get("reason")
    if reason is not None:
        reason = str(reason).strip()

    return {
        "selected_index": selected_index,
        "selected_candidate": selected_candidate,
        "selected_entity_id": selected_candidate.get("entity_id"),
        "selected_label": selected_candidate.get("label"),
        "selected_description": selected_candidate.get("description"),
        "reason": reason,
        "raw_response": raw_response,
        "cache_hit": False,
        "context_signature": context_signature,
        "provider": active_config.provider,
        "model": active_config.model,
    }


def classify_records_with_llm_no_cache(
    records,
    candidate_bank,
    *,
    context_word_window=DEFAULT_LLM_CONTEXT_WORD_WINDOW,
    config=None,
    client=None,
    max_tokens=DEFAULT_LLM_MAX_TOKENS,
):
    llm_client = client or LocalLLMClient(config)
    assignments = []

    for idx, record in enumerate(records, start=1):
        result = choose_wikidata_candidate_with_llm_no_cache(
            span_text=record.get("matched_text") or record.get("query_span") or "",
            context_text=record["context_text"],
            matched_text=record.get("matched_text"),
            local_span=record.get("local_span"),
            candidate_bank=candidate_bank,
            context_word_window=context_word_window,
            config=config,
            client=llm_client,
            max_tokens=max_tokens,
        )
        selected_candidate = result["selected_candidate"]
        assignments.append(
            {
                "record_index": record["record_index"],
                "assigned_description": selected_candidate.get("description"),
                "predicted_label": selected_candidate.get("label"),
                "predicted_entity_id": selected_candidate.get("entity_id"),
                "predicted_definition": selected_candidate.get("definition"),
                "score": None,
                "provenance": "llm_semantic_label_no_cache",
                "llm_reason": result.get("reason"),
                "llm_cache_hit": False,
            }
        )
        if idx % 20 == 0 or idx == len(records):
            print(f"LLM-labeled {idx}/{len(records)} records for {llm_client.config.model}")

    return {
        "assignments": assignments,
        "cache_hits": 0,
        "cache_misses": len(records),
        "provider": llm_client.config.provider,
        "model": llm_client.config.model,
        "total_tokens": llm_client.total_tokens,
    }


def build_assignment_frame(records, assignments, method_name):
    assignment_lookup = {item["record_index"]: item for item in assignments}
    rows = []
    for record in records:
        item = assignment_lookup[record["record_index"]]
        rows.append(
            {
                "record_index": record["record_index"],
                "title": record["title"],
                "kind": record["kind"],
                "matched_text": record["matched_text"],
                "source_text": record["context_text"],
                "assigned_description": item["assigned_description"],
                "predicted_label": item.get("predicted_label"),
                "predicted_entity_id": item.get("predicted_entity_id"),
                "predicted_definition": item.get("predicted_definition"),
                "score": item.get("score"),
                "provenance": item.get("provenance"),
                "llm_reason": item.get("llm_reason"),
                "llm_cache_hit": item.get("llm_cache_hit"),
                "method": method_name,
            }
        )
    return pd.DataFrame(rows).sort_values(["record_index"]).reset_index(drop=True)


def run_llm_experiments(records, candidate_bank, experiments, *, context_word_window, max_tokens):
    results = []
    for experiment in experiments:
        label = experiment["label"]
        provider = experiment["provider"]
        model = experiment["model"]
        api_key_file = experiment.get("api_key_file")

        print(f"\n=== Running {label}: provider={provider}, model={model} ===")
        try:
            llm_config = LocalLLMConfig.from_env(
                provider=provider,
                model=model,
                api_key_file=api_key_file,
            )
            llm_client = LocalLLMClient(llm_config)
            llm_result = classify_records_with_llm_no_cache(
                records,
                candidate_bank,
                context_word_window=context_word_window,
                config=llm_config,
                client=llm_client,
                max_tokens=max_tokens,
            )
            assignment_df = build_assignment_frame(records, llm_result["assignments"], label).rename(
                columns={
                    "assigned_description": label,
                    "llm_reason": f"{label}_reason",
                    "predicted_label": f"{label}_predicted_label",
                    "predicted_entity_id": f"{label}_predicted_entity_id",
                }
            )
            results.append(
                {
                    "label": label,
                    "provider": provider,
                    "model": model,
                    "api_key_file": api_key_file,
                    "error": None,
                    "llm_result": llm_result,
                    "assignment_df": assignment_df,
                }
            )
        except Exception as exc:
            print(f"Failed: {exc}")
            results.append(
                {
                    "label": label,
                    "provider": provider,
                    "model": model,
                    "api_key_file": api_key_file,
                    "error": str(exc),
                    "llm_result": None,
                    "assignment_df": None,
                }
            )
    return results


def build_prediction_wide_df(records, experiment_results):
    base_df = pd.DataFrame(
        [
            {
                "record_index": record["record_index"],
                "title": record["title"],
                "kind": record["kind"],
                "matched_text": record["matched_text"],
                "source_text": record["context_text"],
            }
            for record in records
        ]
    )

    for result in experiment_results:
        if result["assignment_df"] is None:
            continue
        label = result["label"]
        merge_cols = [
            "record_index",
            label,
            f"{label}_reason",
            f"{label}_predicted_label",
            f"{label}_predicted_entity_id",
        ]
        base_df = base_df.merge(result["assignment_df"][merge_cols], on="record_index", how="left")

    return base_df.sort_values("record_index").reset_index(drop=True)


def build_pairwise_agreement_df(prediction_df, labels):
    rows = []
    total = len(prediction_df)
    for left_label, right_label in combinations(labels, 2):
        agreement_count = int((prediction_df[left_label] == prediction_df[right_label]).sum())
        rows.append(
            {
                "model_a": left_label,
                "model_b": right_label,
                "agreement_count": agreement_count,
                "total_records": total,
                "agreement_ratio": agreement_count / total if total else 0.0,
            }
        )
    return pd.DataFrame(rows)


def build_experiment_summary_df(experiment_results):
    rows = []
    for result in experiment_results:
        llm_result = result["llm_result"] or {}
        assignment_df = result["assignment_df"]
        rows.append(
            {
                "label": result["label"],
                "provider": result["provider"],
                "model": result["model"],
                "status": "ok" if result["error"] is None else "error",
                "num_predictions": 0 if assignment_df is None else int(len(assignment_df)),
                "total_tokens": llm_result.get("total_tokens"),
                "error": result["error"],
            }
        )
    return pd.DataFrame(rows)


def build_model_disagreement_df(prediction_df, labels):
    disagreement_mask = prediction_df[labels].nunique(axis=1) > 1
    keep_cols = ["record_index", "title", "kind", "matched_text", "source_text", *labels]
    reason_cols = [f"{label}_reason" for label in labels if f"{label}_reason" in prediction_df.columns]
    keep_cols.extend(reason_cols)
    return prediction_df.loc[disagreement_mask, keep_cols].reset_index(drop=True)

In [4]:
TARGET_SPAN = "director"
QUERY_KIND = DEFAULT_QUERY_KIND
INPUT_SAMPLE_LIMIT = 150  # 可以改成 150，与原 notebook 一致
PROMPT_CONTEXT_MODE = DEFAULT_CONTEXT_MODE
MARK_TARGET = False

WIKIDATA_CANDIDATE_LIMIT = 5
USE_DETAILED_DESCRIPTION = False
REQUIRE_DETAILED_DESCRIPTION = True
EXACT_MATCH_TEXT = True
FILTER_NAME = True

# 为了尽量保持与旧 notebook 接近，这里仍然保留 candidate filtering 开关。
# 但默认关闭它的缓存，避免候选集合直接命中旧结果。
USE_LLM_WIKIDATA = True
LLM_WIKIDATA_USE_API = True
LLM_WIKIDATA_USE_CACHE = False

LLM_CONTEXT_WORD_WINDOW = DEFAULT_LLM_CONTEXT_WORD_WINDOW
LLM_MAX_TOKENS = DEFAULT_LLM_MAX_TOKENS

# 按需增删模型。这里默认给一个本地模型和一个 OpenAI 模型示例。
# 如果某个模型不可用，运行时会记录错误并继续其他模型。
LLM_EXPERIMENTS = [
    {
        "label": "llama3_local",
        "provider": "local",
        "model": "meta-llama/Meta-Llama-3-8B-Instruct",
        "api_key_file": None,
    },
    {
        "label": "gpt54mini_openai",
        "provider": "openai",
        "model": "gpt-5.4-mini",
        "api_key_file": "API_KEY",
    },
]

print(f"Target span: {TARGET_SPAN}")
print(f"Sample limit: {INPUT_SAMPLE_LIMIT}")
print(f"Prompt context mode: {PROMPT_CONTEXT_MODE}")
print(f"LLM semantic cache: disabled")
print(f"LLM Wikidata filter cache: {LLM_WIKIDATA_USE_CACHE}")
display(pd.DataFrame(LLM_EXPERIMENTS))

Target span: director
Sample limit: 150
Prompt context mode: sentence_neighbors
LLM semantic cache: disabled
LLM Wikidata filter cache: False


,label,provider,model,api_key_file
0,llama3_local,local,meta-llama/Meta-Llama-3-8B-Instruct,None
1,gpt54mini_openai,openai,gpt-5.4-mini,API_KEY


In [5]:
embedding_store = load_hotpot_scan_store()

candidate_definitions_df, definition_column = load_wikidata_definition_candidates(
    TARGET_SPAN,
    use_detailed_description=USE_DETAILED_DESCRIPTION,
    exact_match_text=EXACT_MATCH_TEXT,
    filter_name=FILTER_NAME,
    require_detailed_description=REQUIRE_DETAILED_DESCRIPTION,
    candidate_limit=WIKIDATA_CANDIDATE_LIMIT,
    use_llm_filter=USE_LLM_WIKIDATA,
    llm_filter_use_api=LLM_WIKIDATA_USE_API,
    llm_filter_use_cache=LLM_WIKIDATA_USE_CACHE,
)
candidate_bank = build_wikidata_candidate_bank(candidate_definitions_df, definition_column)

query_result = collect_query_span_records(
    embedding_store,
    TARGET_SPAN,
    kind=QUERY_KIND,
    prompt_context_mode=PROMPT_CONTEXT_MODE,
    mark_target=MARK_TARGET,
    max_records=INPUT_SAMPLE_LIMIT,
)
records = query_result["records"]

print(f"Candidate count: {len(candidate_bank)}")
print(f"Record count: {len(records)}")
display(candidate_definitions_df)
display(
    pd.DataFrame(
        [
            {
                "record_index": record["record_index"],
                "title": record["title"],
                "kind": record["kind"],
                "matched_text": record["matched_text"],
                "source_text": record["context_text"],
            }
            for record in records
        ]
    ).head(20)
)

Loaded scan-only store from hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl
{'num_documents': 66581, 'num_unique_terms': 634624, 'num_phrase_occurrences': 1007780, 'num_token_occurrences': 584999, 'num_total_occurrences': 1592779}
Candidate count: 2
Record count: 150


,id,label,description,detailed_description,match_text,aliases,concepturi,source_entity_ids,source_labels,is_merged,is_rewritten
0,Q2526255,film director,a person who directs the artistic or dramatic aspects of a performance or production,a person who directs the artistic or dramatic aspects of a performance or production,,(),,"(Q2526255, Q3387717)","(film director, theatrical director)",True,True
1,Q1162163,director,person who leads a particular area of a company or organization,person who leads a particular area of a company or organization,,(),,"(Q1162163,)","(director,)",False,False


,record_index,title,kind,matched_text,source_text
0,0,Ed Wood,token,director,"Edward Davis Wood Jr. (october 10, 1924 – december 10, 1978) was an american filmmaker, actor, writer, producer, and director."
1,1,David Weissman,token,director,"David Weissman is a screenwriter and director. his film credits include The Family Man (2000), evolution (2001), and when in rome (2010)."
2,2,Ambroise Thomas,token,director,"Charles Louis Ambroise Thomas (5 august 1811 – 12 february 1896) was a french composer, best known for his operas mignon (1866) and hamlet (1868, after shakespeare) and as director of the conservatoire de paris from 1871 till his death."
3,3,Ferdinando Provesi,token,director,"bartolomeo cathedral in busseto (the town very close to Le Roncole, the village where verdi was born.) provesi was also director of the municipal music school and local Philharmonic Society. he began teaching verdi in 1824 when the future composer was 11 years old."
4,4,United States Assistant Secretary of State,token,director,"Assistant Secretaries usually manage individual bureaus of the department of state. when the manager of a bureau or another agency holds a title other than Assistant Secretary, such as director, it can be said to be of Assistant Secretary equivalent rank. Assistant Secretaries typically have a set of deputies, referred to as deputy Assistant Secretaries (DAS)."
5,5,Steven K. Galson,token,director,"he is a retired rear admiral in the United States Public Health Service Commissioned Corps and public health administrator who served as the acting Surgeon General of the United States from october 1, 2007 – october 1, 2009. he served concurrently as acting Assistant Secretary for health from january 22, 2009 to june 25, 2009, and as the Deputy Director and director of the center for Drug Evaluation and research (CDER) at the food and Drug Administration from 2001 to 2007. as the acting Surgeon General, he was the commander of the United States Public Health Service Commissioned Corps and, while serving as the Assistant Secretary for health, was the operational head of the Public Health Service."
6,6,Robert R. Hood,token,director,past roles with the federal government include a role at the White House as Special Assistant to the president in the office of Legislative Affairs and posts at the United States Department of defense as principal deputy assistant secretary for legislative affairs and as deputy under secretary of defense for budget and appropriations affairs. hood was also the director of congressional affairs at the National Nuclear Security Administration.
7,7,Heidi Ewing,token,director,"Heidi Ewing is a director, producer, and writer of documentary films. she and Rachel Grady founded Loki Films in 2001, and have collaborated on several documentaries together."
8,8,Heidi Ewing,token,director,"Norman Lear: Just Another Version of you was the opening night selection of the 2016 Sundance Film Festival and premiered on PBS American Masters on october 25, 2016. other films as a director include The Boys of baraka, freakonomics, and The Education of Mohammed Hussein."
9,9,Antonino Lo Surdo,token,director,"Antonino Lo Surdo (4 february 1880 in syracuse – 7 june 1949 in rome) was an italian physicist. he was appointed as professor of physics at the istituto di fisica in rome in 1919; upon the death of Orso Mario Corbino in 1937, he became the director. Lo Surdo studied terrestrial physics, including seismology and geophysics; the 1908 messina earthquake caused the death of his parents and other close relatives, except his brother."


In [6]:
experiment_results = run_llm_experiments(
    records,
    candidate_bank,
    LLM_EXPERIMENTS,
    context_word_window=LLM_CONTEXT_WORD_WINDOW,
    max_tokens=LLM_MAX_TOKENS,
)

experiment_summary_df = build_experiment_summary_df(experiment_results)
display(experiment_summary_df)

successful_results = [result for result in experiment_results if result["assignment_df"] is not None]
successful_labels = [result["label"] for result in successful_results]

if not successful_results:
    raise RuntimeError("No LLM experiment finished successfully.")

prediction_df = build_prediction_wide_df(records, successful_results)
display(prediction_df.head(20))

if len(successful_labels) >= 2:
    pairwise_agreement_df = build_pairwise_agreement_df(prediction_df, successful_labels)
    display(pairwise_agreement_df)
else:
    print("Only one model succeeded, so pairwise agreement is skipped.")

model_disagreement_df = build_model_disagreement_df(prediction_df, successful_labels)
print(f"Model disagreement rows: {len(model_disagreement_df)}")
display(model_disagreement_df)


=== Running llama3_local: provider=local, model=meta-llama/Meta-Llama-3-8B-Instruct ===
LLM-labeled 20/150 records for meta-llama/Meta-Llama-3-8B-Instruct
LLM-labeled 40/150 records for meta-llama/Meta-Llama-3-8B-Instruct
LLM-labeled 60/150 records for meta-llama/Meta-Llama-3-8B-Instruct
LLM-labeled 80/150 records for meta-llama/Meta-Llama-3-8B-Instruct
LLM-labeled 100/150 records for meta-llama/Meta-Llama-3-8B-Instruct
LLM-labeled 120/150 records for meta-llama/Meta-Llama-3-8B-Instruct
LLM-labeled 140/150 records for meta-llama/Meta-Llama-3-8B-Instruct
LLM-labeled 150/150 records for meta-llama/Meta-Llama-3-8B-Instruct

=== Running gpt54mini_openai: provider=openai, model=gpt-5.4-mini ===
LLM-labeled 20/150 records for gpt-5.4-mini
LLM-labeled 40/150 records for gpt-5.4-mini
LLM-labeled 60/150 records for gpt-5.4-mini
LLM-labeled 80/150 records for gpt-5.4-mini
LLM-labeled 100/150 records for gpt-5.4-mini
LLM-labeled 120/150 records for gpt-5.4-mini
LLM-labeled 140/150 records for gp

,label,provider,model,status,num_predictions,total_tokens,error
0,llama3_local,local,meta-llama/Meta-Llama-3-8B-Instruct,ok,150,48794,None
1,gpt54mini_openai,openai,gpt-5.4-mini,ok,150,47636,None


,record_index,title,kind,matched_text,source_text,llama3_local,llama3_local_reason,llama3_local_predicted_label,llama3_local_predicted_entity_id,gpt54mini_openai,gpt54mini_openai_reason,gpt54mini_openai_predicted_label,gpt54mini_openai_predicted_entity_id
0,0,Ed Wood,token,director,"Edward Davis Wood Jr. (october 10, 1924 – december 10, 1978) was an american filmmaker, actor, writer, producer, and director.",a person who directs the artistic or dramatic aspects of a performance or production,"The context mentions 'filmmaker' and 'actor, writer, producer, and director', indicating a strong connection to the film industry, making 'film director' the most semantically fitting candidate.",film director,Q2526255,a person who directs the artistic or dramatic aspects of a performance or production,"The context describes Edward Wood as an American filmmaker, actor, writer, producer, and director, so 'director' refers to a film director.",film director,Q2526255
1,1,David Weissman,token,director,"David Weissman is a screenwriter and director. his film credits include The Family Man (2000), evolution (2001), and when in rome (2010).",a person who directs the artistic or dramatic aspects of a performance or production,"The context mentions 'film credits' and 'screenwriter', which suggests a strong connection to the film industry, making 'film director' a more semantically fitting choice.",film director,Q2526255,a person who directs the artistic or dramatic aspects of a performance or production,"The context describes David Weissman as a screenwriter and mentions film credits, so 'director' refers to a film director rather than an organizational director.",film director,Q2526255
2,2,Ambroise Thomas,token,director,"Charles Louis Ambroise Thomas (5 august 1811 – 12 february 1896) was a french composer, best known for his operas mignon (1866) and hamlet (1868, after shakespeare) and as director of the conservatoire de paris from 1871 till his death.",a person who directs the artistic or dramatic aspects of a performance or production,"The context mentions the director of the Conservatoire de Paris, which is a music school, and the director is likely referring to a person who directs the artistic or dramatic aspects of a performance or production, which is the definition of a film director.",film director,Q2526255,person who leads a particular area of a company or organization,"The context says he was director of the Conservatoire de Paris, which is an organizational leadership role, not a film/production director.",director,Q1162163
3,3,Ferdinando Provesi,token,director,"bartolomeo cathedral in busseto (the town very close to Le Roncole, the village where verdi was born.) provesi was also director of the municipal music school and local Philharmonic Society. he began teaching verdi in 1824 when the future composer was 11 years old.",person who leads a particular area of a company or organization,"The context suggests a director of an organization or institution, which is a better fit for the definition of a director (Q1162163) than a film director (Q2526255).",director,Q1162163,person who leads a particular area of a company or organization,"The context says he was director of the municipal music school and local Philharmonic Society, so it refers to an organizational leader, not a film director.",director,Q1162163
4,4,United States Assistant Secretary of State,token,director,"Assistant Secretaries usually manage individual bureaus of the department of state. when the manager of a bureau or another agency holds a title other than Assistant Secretary, such as director, it can be said to be of Assistant Secretary equivalent rank. Assistant Secretaries typically have a set of deputies, referred to as deputy Assistant Secretaries (DAS).",person who leads a particular area of a company or organization,"The context suggests a director as a leader of a particular area of a company or organization, which is a more fitting semantic match than a film d

,model_a,model_b,agreement_count,total_records,agreement_ratio
0,llama3_local,gpt54mini_openai,134,150,0.893333


Model disagreement rows: 16


,record_index,title,kind,matched_text,source_text,llama3_local,gpt54mini_openai,llama3_local_reason,gpt54mini_openai_reason
0,2,Ambroise Thomas,token,director,"Charles Louis Ambroise Thomas (5 august 1811 – 12 february 1896) was a french composer, best known for his operas mignon (1866) and hamlet (1868, after shakespeare) and as director of the conservatoire de paris from 1871 till his death.",a person who directs the artistic or dramatic aspects of a performance or production,person who leads a particular area of a company or organization,"The context mentions the director of the Conservatoire de Paris, which is a music school, and the director is likely referring to a person who directs the artistic or dramatic aspects of a performance or production, which is the definition of a film director.","The context says he was director of the Conservatoire de Paris, which is an organizational leadership role, not a film/production director."
1,6,Robert R. Hood,token,director,past roles with the federal government include a role at the White House as Special Assistant to the president in the office of Legislative Affairs and posts at the United States Department of defense as principal deputy assistant secretary for legislative affairs and as deputy under secretary of defense for budget and appropriations affairs. hood was also the director of congressional affairs at the National Nuclear Security Administration.,a person who directs the artistic or dramatic aspects of a performance or production,person who leads a particular area of a company or organization,"The context suggests a role in the film or performance industry, which aligns with the definition of a film director.","The context refers to a government/organizational role as director of congressional affairs, so it matches a person who leads a particular area of an organization, not a film director."
2,76,Herbert Brean,token,director,"Herbert Brean (10 december 1907 – may 7, 1973) was an american journalist and crime fiction writer, best known for his recurring series characters William Deacon and Reynold Frame. he was a director and former executive vice president of the Mystery Writers of america, a group for which he also taught a class in mystery writing. aside from his seven mystery crime novels, he also published non-fiction books and articles, and mystery magazine short stories.",a person who directs the artistic or dramatic aspects of a performance or production,person who leads a particular area of a company or organization,"The context mentions a director of a performance or production, which is a film director, and not a company or organization director.","The context says he was a director and former executive vice president of the Mystery Writers of America, which indicates an organizational leadership role, not a film or stage directing role."
3,78,Ted Plumb,token,director,"Louis Rams. teddy served as the Wide Receivers Coach with the monsters of the midway Chicago Bears, and he served as the director of Pro Scouting for the Greatest Show on turf 2000 st. Louis Rams (according to the st.",a person who directs the artistic or dramatic aspects of a performance or production,person who leads a particular area of a company or organization,"The context suggests a creative or artistic role, which is more closely related to a film director than a corporate director.","The context says he served as the director of Pro Scouting for the St. Louis Rams, which is a leadership role within an organization, not a film/production role."
4,81,Lori L. Holt,token,director,"in cognitive psychology with a minor in neurophysiology from UW–madison in 1999, and she has been employed at Carnegie Mellon University and has been a member of the center for the Neural Basis of cognition ever since. holt is the director of the Speech Perception & Learning Laboratory at Carnegie Mellon University. she was one of two recipients of the Troland Research Awards in 2013.",a person who directs th

In [7]:
def export_experiment_results(
    output_dir,
    *,
    target_span,
    query_kind,
    input_sample_limit,
    prompt_context_mode,
    mark_target,
    wikidata_candidate_limit,
    use_detailed_description,
    require_detailed_description,
    exact_match_text,
    filter_name,
    use_llm_wikidata,
    llm_wikidata_use_api,
    llm_wikidata_use_cache,
    llm_context_word_window,
    llm_max_tokens,
    llm_experiments,
    experiment_summary_df,
    pairwise_agreement_df,
    model_disagreement_df,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    parameter_rows = [
        ("target_span", target_span),
        ("query_kind", query_kind),
        ("input_sample_limit", input_sample_limit),
        ("prompt_context_mode", prompt_context_mode),
        ("mark_target", mark_target),
        ("wikidata_candidate_limit", wikidata_candidate_limit),
        ("use_detailed_description", use_detailed_description),
        ("require_detailed_description", require_detailed_description),
        ("exact_match_text", exact_match_text),
        ("filter_name", filter_name),
        ("use_llm_wikidata", use_llm_wikidata),
        ("llm_wikidata_use_api", llm_wikidata_use_api),
        ("llm_wikidata_use_cache", llm_wikidata_use_cache),
        ("llm_semantic_cache", False),
        ("llm_context_word_window", llm_context_word_window),
        ("llm_max_tokens", llm_max_tokens),
    ]
    parameter_df = pd.DataFrame(parameter_rows, columns=["parameter", "value"])
    parameter_df.to_csv(output_dir / "experiment_params.csv", index=False)

    pd.DataFrame(llm_experiments).to_csv(output_dir / "model_configs.csv", index=False)
    experiment_summary_df.to_csv(output_dir / "experiment_summary.csv", index=False)

    if pairwise_agreement_df is not None and not pairwise_agreement_df.empty:
        pairwise_agreement_df.to_csv(output_dir / "pairwise_agreement.csv", index=False)

    disagreement_export_df = model_disagreement_df.drop(
        columns=[col for col in ["record_index", "title"] if col in model_disagreement_df.columns]
    )
    disagreement_export_df.to_csv(output_dir / "model_disagreements.csv", index=False)
    (output_dir / "model_disagreements.txt").write_text(
        disagreement_export_df.to_string(index=False),
        encoding="utf-8",
    )

    summary_lines = [
        f"target_span: {target_span}",
        f"input_sample_limit: {input_sample_limit}",
        f"prompt_context_mode: {prompt_context_mode}",
        "llm_semantic_cache: disabled",
        f"llm_wikidata_use_cache: {llm_wikidata_use_cache}",
        f"num_models: {len(llm_experiments)}",
        f"num_disagreement_rows: {len(disagreement_export_df)}",
    ]
    (output_dir / "README.txt").write_text("\n".join(summary_lines), encoding="utf-8")
    print(f"Saved experiment results to {output_dir}")
    return disagreement_export_df


OUTPUT_DIR = Path("logs/hotpotqa_llm_model_compare_director")
pairwise_agreement_result_df = pairwise_agreement_df if "pairwise_agreement_df" in globals() else pd.DataFrame()
saved_disagreement_df = export_experiment_results(
    OUTPUT_DIR,
    target_span=TARGET_SPAN,
    query_kind=QUERY_KIND,
    input_sample_limit=INPUT_SAMPLE_LIMIT,
    prompt_context_mode=PROMPT_CONTEXT_MODE,
    mark_target=MARK_TARGET,
    wikidata_candidate_limit=WIKIDATA_CANDIDATE_LIMIT,
    use_detailed_description=USE_DETAILED_DESCRIPTION,
    require_detailed_description=REQUIRE_DETAILED_DESCRIPTION,
    exact_match_text=EXACT_MATCH_TEXT,
    filter_name=FILTER_NAME,
    use_llm_wikidata=USE_LLM_WIKIDATA,
    llm_wikidata_use_api=LLM_WIKIDATA_USE_API,
    llm_wikidata_use_cache=LLM_WIKIDATA_USE_CACHE,
    llm_context_word_window=LLM_CONTEXT_WORD_WINDOW,
    llm_max_tokens=LLM_MAX_TOKENS,
    llm_experiments=LLM_EXPERIMENTS,
    experiment_summary_df=experiment_summary_df,
    pairwise_agreement_df=pairwise_agreement_result_df,
    model_disagreement_df=model_disagreement_df,
)
display(saved_disagreement_df.head(20))


Saved experiment results to logs/hotpotqa_llm_model_compare_director


,kind,matched_text,source_text,llama3_local,gpt54mini_openai,llama3_local_reason,gpt54mini_openai_reason
0,token,director,"Charles Louis Ambroise Thomas (5 august 1811 – 12 february 1896) was a french composer, best known for his operas mignon (1866) and hamlet (1868, after shakespeare) and as director of the conservatoire de paris from 1871 till his death.",a person who directs the artistic or dramatic aspects of a performance or production,person who leads a particular area of a company or organization,"The context mentions the director of the Conservatoire de Paris, which is a music school, and the director is likely referring to a person who directs the artistic or dramatic aspects of a performance or production, which is the definition of a film director.","The context says he was director of the Conservatoire de Paris, which is an organizational leadership role, not a film/production director."
1,token,director,past roles with the federal government include a role at the White House as Special Assistant to the president in the office of Legislative Affairs and posts at the United States Department of defense as principal deputy assistant secretary for legislative affairs and as deputy under secretary of defense for budget and appropriations affairs. hood was also the director of congressional affairs at the National Nuclear Security Administration.,a person who directs the artistic or dramatic aspects of a performance or production,person who leads a particular area of a company or organization,"The context suggests a role in the film or performance industry, which aligns with the definition of a film director.","The context refers to a government/organizational role as director of congressional affairs, so it matches a person who leads a particular area of an organization, not a film director."
2,token,director,"Herbert Brean (10 december 1907 – may 7, 1973) was an american journalist and crime fiction writer, best known for his recurring series characters William Deacon and Reynold Frame. he was a director and former executive vice president of the Mystery Writers of america, a group for which he also taught a class in mystery writing. aside from his seven mystery crime novels, he also published non-fiction books and articles, and mystery magazine short stories.",a person who directs the artistic or dramatic aspects of a performance or production,person who leads a particular area of a company or organization,"The context mentions a director of a performance or production, which is a film director, and not a company or organization director.","The context says he was a director and former executive vice president of the Mystery Writers of America, which indicates an organizational leadership role, not a film or stage directing role."
3,token,director,"Louis Rams. teddy served as the Wide Receivers Coach with the monsters of the midway Chicago Bears, and he served as the director of Pro Scouting for the Greatest Show on turf 2000 st. Louis Rams (according to the st.",a person who directs the artistic or dramatic aspects of a performance or production,person who leads a particular area of a company or organization,"The context suggests a creative or artistic role, which is more closely related to a film director than a corporate director.","The context says he served as the director of Pro Scouting for the St. Louis Rams, which is a leadership role within an organization, not a film/production role."
4,token,director,"in cognitive psychology with a minor in neurophysiology from UW–madison in 1999, and she has been employed at Carnegie Mellon University and has been a member of the center for the Neural Basis of cognition ever since. holt is the director of the Speech Perception & Learning Laboratory at Carnegie Mellon University. she was one of two recipients of the Troland Research Awards in 2013.",a person who directs the artistic or dramatic aspects of a performance or production,person who leads a particular area of 

In [8]:
# Export is already handled in the previous cell.
# This cell is intentionally left as a no-op placeholder.
